# Gộp keypoints từ 2 part

Gắn cả 2 part, và với mỗi part gắn đủ 2 version, vào Input của Kaggle. Notebook quét mọi thư mục `keypoints_raw` dưới `/kaggle/input`.

Gloss trùng tên folder được gộp thành một folder. Cùng `split/gloss/tên_file.npy` chỉ giữ một bản: file lớn hơn được giữ, file cùng kích thước bị bỏ. Bản chung là hai folder `/kaggle/working/train` và `/kaggle/working/test`.

Cell cuối tạo dataset private mới từ hai folder đó. Bật Internet, rồi thêm secret `KAGGLE_USERNAME` và `KAGGLE_KEY`.


In [ ]:
import logging
import os
import shutil
import unicodedata
from collections import defaultdict
from multiprocessing import get_context
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    force=True,
)
logger = logging.getLogger("merge_keypoints")

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
SPLITS = ("train", "test")
SKIP_DIR_NAMES = {"mediapipe_models"}
EXPECTED_ROOTS = 4


def canonical_name(name: str) -> str:
    return unicodedata.normalize("NFC", name)


def find_keypoint_roots(input_dir: Path) -> list[Path]:
    # Each attached dataset version is its own mount. Stop at keypoints_raw
    # so the walk does not descend into every gloss folder.
    roots = []
    if not input_dir.is_dir():
        logger.error("Input directory does not exist: %s", input_dir)
        return roots

    for dirpath, dirnames, _filenames in os.walk(input_dir):
        dirnames[:] = [
            name
            for name in dirnames
            if name not in SKIP_DIR_NAMES and not name.startswith(".")
        ]
        if os.path.basename(dirpath) != "keypoints_raw":
            continue
        has_split = any(
            os.path.isdir(os.path.join(dirpath, split)) for split in SPLITS
        )
        if has_split:
            roots.append(Path(dirpath))
        dirnames.clear()

    roots.sort(key=lambda path: str(path))
    return roots


def index_root(root: Path) -> dict[tuple[str, str, str], tuple[int, str]]:
    indexed = {}
    for split in SPLITS:
        split_dir = root / split
        if not split_dir.is_dir():
            continue
        with os.scandir(split_dir) as glosses:
            for gloss in glosses:
                if not gloss.is_dir(follow_symlinks=False):
                    continue
                gloss_name = canonical_name(gloss.name)
                with os.scandir(gloss.path) as samples:
                    for sample in samples:
                        if not sample.is_file(follow_symlinks=False):
                            continue
                        if not sample.name.endswith(".npy"):
                            continue
                        key = (split, gloss_name, canonical_name(sample.name))
                        indexed[key] = (
                            sample.stat(follow_symlinks=False).st_size,
                            sample.path,
                        )
    return indexed


def _link_or_copy(job: tuple[str, str]) -> str:
    src, dst = job
    try:
        os.link(src, dst)
        return "link"
    except OSError:
        shutil.copy2(src, dst)
        return "copy"


if INPUT_DIR.is_dir():
    logger.info("Mounted inputs: %s", sorted(p.name for p in INPUT_DIR.iterdir()))
else:
    logger.error("Nothing is mounted at %s. Attach every part and both versions.", INPUT_DIR)

roots = find_keypoint_roots(INPUT_DIR)
for index, root in enumerate(roots):
    logger.info("Source %d: %s", index, root)

if len(roots) < 2:
    raise RuntimeError(
        f"Found {len(roots)} keypoints_raw tree(s). Attach both parts, and both versions of each."
    )
if len(roots) != EXPECTED_ROOTS:
    logger.warning(
        "Expected %d keypoints_raw trees (2 parts x 2 versions), found %d.",
        EXPECTED_ROOTS,
        len(roots),
    )


In [ ]:
indexed_by_root = []
worker_count = min(len(roots), max(1, os.cpu_count() or 1))
with get_context("fork").Pool(processes=worker_count) as pool:
    indexed_by_root = pool.map(index_root, roots, chunksize=1)

# key -> (size, source_path, source_index)
chosen: dict[tuple[str, str, str], tuple[int, str, int]] = {}
gloss_sources: dict[tuple[str, str], set[int]] = defaultdict(set)
stats = {
    "empty_skipped": 0,
    "duplicate_same_size": 0,
    "replaced_by_larger": 0,
    "smaller_dropped": 0,
}

for source_index, indexed in enumerate(indexed_by_root):
    glosses = {key[1] for key in indexed}
    logger.info(
        "Source %d indexed: %d npy files, %d glosses, root=%s",
        source_index,
        len(indexed),
        len(glosses),
        roots[source_index],
    )
    for key, (size, path) in indexed.items():
        split, gloss, _name = key
        gloss_sources[(split, gloss)].add(source_index)
        if size <= 0:
            stats["empty_skipped"] += 1
            logger.warning("Skip empty file: %s", path)
            continue
        current = chosen.get(key)
        if current is None:
            chosen[key] = (size, path, source_index)
            continue
        current_size, _current_path, _current_source = current
        if size == current_size:
            stats["duplicate_same_size"] += 1
            continue
        if size > current_size:
            chosen[key] = (size, path, source_index)
            stats["replaced_by_larger"] += 1
            logger.warning(
                "Same name, different size, keep larger: %s/%s/%s (%d > %d)",
                split,
                gloss,
                key[2],
                size,
                current_size,
            )
            continue
        stats["smaller_dropped"] += 1

overlapping = sorted(
    (split, gloss, sources)
    for (split, gloss), sources in gloss_sources.items()
    if len(sources) > 1
)
kept_bytes = sum(size for size, _path, _source in chosen.values())
logger.info("Unique files kept: %d (%d bytes)", len(chosen), kept_bytes)
logger.info("Duplicate same-size dropped: %d", stats["duplicate_same_size"])
logger.info("Replaced by a larger copy: %d", stats["replaced_by_larger"])
logger.info("Smaller copies dropped: %d", stats["smaller_dropped"])
logger.info("Empty files skipped: %d", stats["empty_skipped"])
logger.info("Overlapping glosses: %d", len(overlapping))
for split, gloss, sources in overlapping:
    logger.info("Overlap %s/%s from sources %s", split, gloss, sorted(sources))


In [ ]:
from tqdm import tqdm

for split in SPLITS:
    split_dir = OUTPUT_DIR / split
    if split_dir.exists():
        shutil.rmtree(split_dir)
        logger.info("Removed previous %s", split_dir)

jobs = []
dest_dirs = set()
for (split, gloss, name), (_size, src, _source_index) in chosen.items():
    dst = OUTPUT_DIR / split / gloss / name
    dest_dirs.add(dst.parent)
    jobs.append((src, str(dst)))

for dest_dir in dest_dirs:
    dest_dir.mkdir(parents=True, exist_ok=True)

worker_count = max(1, os.cpu_count() or 1)
link_count = 0
copy_count = 0
logger.info("Writing %d files with %d workers", len(jobs), worker_count)
if jobs:
    with get_context("fork").Pool(processes=worker_count) as pool:
        for kind in tqdm(
            pool.imap_unordered(_link_or_copy, jobs, chunksize=32),
            total=len(jobs),
            desc="merge",
        ):
            if kind == "link":
                link_count += 1
            else:
                copy_count += 1

logger.info("Placed %d hardlinks and %d copies into %s", link_count, copy_count, OUTPUT_DIR)


In [ ]:
import numpy as np

merged_counts: dict[tuple[str, str], int] = defaultdict(int)
bad_files = []
checked = 0

for split in SPLITS:
    split_dir = OUTPUT_DIR / split
    if not split_dir.is_dir():
        continue
    for gloss_dir in sorted(path for path in split_dir.iterdir() if path.is_dir()):
        files = sorted(path for path in gloss_dir.glob("*.npy") if path.is_file())
        merged_counts[(split, gloss_dir.name)] = len(files)
        for sample in files[:1]:
            array = np.load(sample, mmap_mode="r")
            checked += 1
            if array.ndim != 3 or array.shape[1:] != (76, 3) or array.shape[0] == 0:
                bad_files.append((str(sample), tuple(int(dim) for dim in array.shape)))

logger.info("Merged glosses: %d", len(merged_counts))
for (split, gloss), count in sorted(merged_counts.items()):
    logger.info("%s/%s: %d files", split, gloss, count)
logger.info("Shape checks: %d", checked)
if bad_files:
    for path, shape in bad_files:
        logger.error("Unexpected array %s shape=%s", path, shape)
    raise RuntimeError(f"{len(bad_files)} merged files are not (T, 76, 3)")
logger.info("Checked arrays are (T, 76, 3).")

expected = len(chosen)
actual = sum(merged_counts.values())
if actual != expected:
    raise RuntimeError(f"File count mismatch: planned {expected}, on disk {actual}")
logger.info("Merged tree matches the plan: %d files", actual)


In [ ]:
import json
import os
import subprocess

DATASET_SLUG = "vsl400-keypoints-merged"
DATASET_TITLE = "VSL400 keypoints merged"

for split in SPLITS:
    if not (OUTPUT_DIR / split).is_dir():
        raise RuntimeError(f"Missing merged folder: {OUTPUT_DIR / split}")

try:
    from kaggle_secrets import UserSecretsClient

    secret_client = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = secret_client.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = secret_client.get_secret("KAGGLE_KEY")
except Exception as exc:
    raise RuntimeError(
        "Add notebook secrets KAGGLE_USERNAME and KAGGLE_KEY, and enable Internet."
    ) from exc

dataset_id = f"{os.environ['KAGGLE_USERNAME'].strip()}/{DATASET_SLUG}"
metadata_path = OUTPUT_DIR / "dataset-metadata.json"
metadata_path.write_text(
    json.dumps(
        {
            "title": DATASET_TITLE,
            "id": dataset_id,
            "licenses": [{"name": "CC0-1.0"}],
        },
        indent=2,
    )
)


def _run_kaggle(args: list[str]) -> int:
    logger.info("kaggle %s", " ".join(args))
    completed = subprocess.run(
        ["kaggle", *args],
        check=False,
        capture_output=True,
        text=True,
    )
    for line in completed.stdout.splitlines():
        logger.info("%s", line)
    for line in completed.stderr.splitlines():
        logger.warning("%s", line)
    return completed.returncode


try:
    # dir-mode zip is only the upload transport. Kaggle unpacks the archives,
    # so the dataset shows train/ and test/ folders.
    create_code = _run_kaggle(
        [
            "datasets",
            "create",
            "-p",
            str(OUTPUT_DIR),
            "--dir-mode",
            "zip",
            "--ignore-patterns",
            "dataset-metadata.json",
            "-q",
        ]
    )
    if create_code != 0:
        logger.info("Dataset %s already exists, pushing a new version.", dataset_id)
        version_code = _run_kaggle(
            [
                "datasets",
                "version",
                "-p",
                str(OUTPUT_DIR),
                "--dir-mode",
                "zip",
                "--ignore-patterns",
                "dataset-metadata.json",
                "-m",
                "Merged train and test keypoints",
                "-q",
            ]
        )
        if version_code != 0:
            raise RuntimeError(f"Could not create or update dataset {dataset_id}")

    _run_kaggle(["datasets", "status", dataset_id])
    logger.info("Dataset: https://www.kaggle.com/datasets/%s", dataset_id)
finally:
    if metadata_path.exists():
        metadata_path.unlink()
    for split in SPLITS:
        archive = OUTPUT_DIR / f"{split}.zip"
        if archive.exists():
            archive.unlink()
            logger.info("Removed upload archive %s", archive)
